# Projeto Interconexão - Modelos estatísticos

A documentação do projeto pode ser encontrada em [Modelo_Interconexão_BRAIN_Guilherme_Andrade](https://algarnet.sharepoint.com/sites/TriboAnalytics246/Documentos%20Compartilhados/Forms/AllItems.aspx?id=%2Fsites%2FTriboAnalytics246%2FDocumentos%20Compartilhados%2FGeneral%2FSquad%20Data%20Wizards%20%2D%20Projetos%2FModelo%5FInterconex%C3%A3o%5FBRAIN%5FGuilherme%5FAndrade&viewid=18eaba8b%2D6b2b%2D40e3%2D85d6%2De3c0e7598b0c&FolderCTID=0x012000E7FCBA090D95C348A4A8E9F05538F026)

Nesse *pipeline* serão realizadas as seguintes ações:

1. Carregamento da tabela sem intervalo de tempo de definido
2. Conversão do timestamp
3. Análise para Janela fixa
4. Análise para Janela móvel


In [ ]:
# === Bibliotecas necessárias
from snowflake.snowpark import Session
import pandas as pd
from snowflake.snowpark import Window
from snowflake.snowpark.functions import col, lit, avg, date_part, sum as ssum
from snowflake.snowpark.functions import count, min, dateadd, lit, unix_timestamp, sql_expr
from snowflake.snowpark import DataFrame

# === Configurações
warnings.filterwarnings("ignore")
session = get_active_session()

In [ ]:
# === Carregar dataframe
df_all = session.table("DWDEV.DATASCIENCE.F_VOICETRAFFIC_SBC_NOTIME")
df_all.show()

In [ ]:
# === Convertar o timestamp em minutos 
df_all = df_all.with_column("epoch_hour",
                   date_part("EPOCH_SECOND", col("DATAHORA")) / 3600)
df_all

### Análise para Janela Fixa (TCR)

In [ ]:
def analyze_recovery_without_blocking_first_only(df_all: DataFrame, 
                                               error_status: int,
                                               collection_window_hours: int,
                                               min_errors_threshold: int = 3,
                                               user_col: str = "IDENTIFICATIONNUMBER_B_11",
                                               datetime_col: str = "DATAHORA",
                                               status_col: str = "SIPSTATUSCODE_135") -> DataFrame:
    """
    Analisa recuperação apenas na PRIMEIRA vez que o usuário atinge o threshold de erros.
    """
    
    # === Filtrar erros e agrupar por usuário e dia
    df_errors = (
        df_all.filter(col(status_col) == error_status)
              .select(col(user_col), col(datetime_col))
              .with_column("TIME_START", sql_expr(f"DATE_TRUNC('DAY', {datetime_col})"))
    )
    
    # === Contar erros por usuário/dia e filtrar pelo threshold
    triggered_users = (
        df_errors.group_by(user_col, "TIME_START")
                 .agg(count("*").alias("error_count"))
                 .filter(col("error_count") >= min_errors_threshold)
                 .with_column("COLLECTION_END", 
                              dateadd("HOUR", lit(collection_window_hours), col("TIME_START")))
    )
    
    # === Pegar apenas a PRIMEIRA ocorrência por usuário (usando window function)
    from snowflake.snowpark.window import Window
    from snowflake.snowpark.functions import row_number
    
    window_spec = Window.partition_by(col(user_col)).order_by(col("TIME_START"))
    
    first_triggered_users = (
        triggered_users
        .with_column("row_number", row_number().over(window_spec))
        .filter(col("row_number") == 1)
        .drop("row_number")
    )
    
    # === Filtrar apenas status 200
    df_success = (
        df_all.filter(col(status_col) == 200)
              .select(col(user_col), col(datetime_col))
    )
    
    # === Join para encontrar 200s que ocorrem após o fim da janela de coleta
    recovery_candidates = first_triggered_users.join(
        df_success,
        first_triggered_users[user_col] == df_success[user_col],
        how="inner"
    ).filter(
        df_success[datetime_col] >= first_triggered_users["COLLECTION_END"]
    )
    
    # === Encontrar primeiro 200 após cada janela de coleta
    first_success = (
        recovery_candidates.group_by(
            first_triggered_users[user_col], 
            first_triggered_users["TIME_START"],
            first_triggered_users["COLLECTION_END"]
        ).agg(
            min(df_success[datetime_col]).alias(f"FIRST_200_AFTER_{error_status}")
        )
    )
    
    # === Calcular tempo em horas até o primeiro 200
    result = first_success.with_column(
        "HOURS_TO_FIRST_200",
        (unix_timestamp(col(f"FIRST_200_AFTER_{error_status}")) - unix_timestamp(col("COLLECTION_END"))) / 3600
    )
    
    return result.select(
        first_triggered_users[user_col].alias("IDENTIFICATIONNUMBER_B_11"),
        first_triggered_users["TIME_START"],
        first_triggered_users["COLLECTION_END"],
        col(f"FIRST_200_AFTER_{error_status}"),
        col("HOURS_TO_FIRST_200")
    )

In [ ]:
def calculate_recovery_metrics(result: DataFrame, df_all: DataFrame,
                               error_status: int,
                               user_col: str = "IDENTIFICATIONNUMBER_B_11",
                               status_col: str = "SIPSTATUSCODE_135") -> DataFrame:
    """
    Calcula métricas estatísticas do tempo de recuperação.

    Args:
        result: DataFrame resultado da função analyze_blocking_recovery
        df_all: DataFrame original com todos os usuários
        user_col: Nome da coluna de identificação do usuário

    Returns:
        DataFrame com métricas: MEAN, MEDIAN, PERCENTILE_90, PERCENTILE_95, PERCENTILE_99,
        TOTAL_RECOVERIES, TOTAL_USERS, BLOCKED_USERS, TOTAL_PCT_BLOCKED
    """

    # Métricas de tempo de recuperação
    metrics_df = result.select(
        avg(col("HOURS_TO_FIRST_200")).alias("MEAN_HOURS_TO_FIRST_200"),
        sql_expr("APPROX_PERCENTILE(HOURS_TO_FIRST_200, 0.5)").alias("MEDIAN_HOURS_TO_FIRST_200"),
        sql_expr("APPROX_PERCENTILE(HOURS_TO_FIRST_200, 0.1)").alias("PERCENTILE_90"),
        sql_expr("APPROX_PERCENTILE(HOURS_TO_FIRST_200, 0.05)").alias("PERCENTILE_95"),
        sql_expr("APPROX_PERCENTILE(HOURS_TO_FIRST_200, 0.01)").alias("PERCENTILE_99"),
        count(col("HOURS_TO_FIRST_200")).alias("TOTAL_RECOVERIES")
    )

    # Total de usuários únicos na base
    total_users_with_error = (
        df_all.filter(col(status_col) == error_status).select(user_col).distinct().count()
                     )

    # Total de usuários que foram bloqueados (presentes no result)
    blocked_users = result.select(user_col).distinct().count()

    # Percentual de usuários bloqueados
    pct_blocked = (blocked_users / total_users_with_error * 100.0) if total_users_with_error > 0 else 0.0

    # Adiciona essas métricas ao DataFrame
    metrics_df = metrics_df.with_column("TOTAL_USERS_WITH_ERROR", lit(total_users_with_error)) \
                           .with_column("BLOCKED_USERS", lit(blocked_users)) \
                           .with_column("TOTAL_PCT_BLOCKED", lit(pct_blocked))

    return metrics_df


In [ ]:
# === Aplicação da função
result_1 = analyze_recovery_without_blocking_first_only(
    df_all=df_all,
    error_status=487,
    collection_window_hours=1,
    min_errors_threshold=1
)

metrics_1 = calculate_recovery_metrics(result_1, df_all, 487)
metrics_1.show()

result_2 = analyze_recovery_without_blocking_first_only(
    df_all=df_all,
    error_status=487,
    collection_window_hours=2,
    min_errors_threshold=1
)

metrics_2 = calculate_recovery_metrics(result_2, df_all, 487)
metrics_2.show()

### Análise para Janela Móvel (OKTOR)

In [ ]:
def analyze_recovery_sliding_window_first_only(df_all: DataFrame, 
                                               error_status: int,
                                               collection_window_hours: int,
                                               min_errors_threshold: int = 3,
                                               user_col: str = "IDENTIFICATIONNUMBER_B_11",
                                               datetime_col: str = "DATAHORA",
                                               status_col: str = "SIPSTATUSCODE_135") -> DataFrame:
    """
    Analisa recuperação com janela móvel: a janela de coleta começa no PRIMEIRO erro
    e considera apenas a PRIMEIRA vez que o usuário atinge o threshold.
    """
    from snowflake.snowpark.window import Window
    from snowflake.snowpark.functions import (
        row_number, when, lit, count, min as min_func, 
        unix_timestamp, dateadd, col, first_value
    )
    
    # === 1. Filtrar erros e ordenar por usuário e timestamp
    df_errors = (
        df_all.filter(col(status_col) == error_status)
              .select(col(user_col), col(datetime_col))
              .order_by(col(user_col), col(datetime_col))
    )
    
    # === 2. Identificar o PRIMEIRO erro de cada usuário usando first_value
    window_first_error = Window.partition_by(col(user_col)).order_by(col(datetime_col))
    
    df_errors_with_window = (
        df_errors
        .with_column("FIRST_ERROR_TIME", 
                     first_value(col(datetime_col)).over(window_first_error))
    )
    
    # === 4. Definir janela móvel baseada no primeiro erro
    df_errors_windowed = (
        df_errors_with_window
        .with_column("COLLECTION_END", 
                     dateadd("HOUR", lit(collection_window_hours), col("FIRST_ERROR_TIME")))
        .filter(col(datetime_col) <= col("COLLECTION_END"))  # Apenas erros dentro da janela
    )
    
    # === 5. Contar erros dentro da janela móvel e filtrar pelo threshold
    triggered_users = (
        df_errors_windowed
        .group_by(col(user_col), col("FIRST_ERROR_TIME"), col("COLLECTION_END"))
        .agg(count("*").alias("error_count"))
        .filter(col("error_count") >= min_errors_threshold)
    )
    
    # === 6. Para usuários com múltiplas janelas triggered, pegar apenas a PRIMEIRA
    window_first_trigger = Window.partition_by(col(user_col)).order_by(col("FIRST_ERROR_TIME"))
    
    first_triggered_users = (
        triggered_users
        .with_column("trigger_rank", row_number().over(window_first_trigger))
        .filter(col("trigger_rank") == 1)
        .drop("trigger_rank")
    )
    
    # === 7. Filtrar apenas status 200 (sucessos)
    df_success = (
        df_all.filter(col(status_col) == 200)
              .select(col(user_col), col(datetime_col))
    )
    
    # === 8. Join para encontrar 200s que ocorrem após o fim da janela de coleta
    recovery_candidates = first_triggered_users.join(
        df_success,
        first_triggered_users[user_col] == df_success[user_col],
        how="inner"
    ).filter(
        df_success[datetime_col] >= first_triggered_users["COLLECTION_END"]
    )
    
    # === 9. Encontrar primeiro 200 após cada janela de coleta
    first_success = (
        recovery_candidates.group_by(
            first_triggered_users[user_col], 
            first_triggered_users["FIRST_ERROR_TIME"],
            first_triggered_users["COLLECTION_END"]
        ).agg(
            min_func(df_success[datetime_col]).alias(f"FIRST_200_AFTER_{error_status}")
        )
    )
    
    # === 10. Calcular tempo em horas até o primeiro 200
    result = first_success.with_column(
        "HOURS_TO_FIRST_200",
        (unix_timestamp(col(f"FIRST_200_AFTER_{error_status}")) - unix_timestamp(col("COLLECTION_END"))) / 3600
    )
    
    return result.select(
        first_triggered_users[user_col].alias("IDENTIFICATIONNUMBER_B_11"),
        col("FIRST_ERROR_TIME").alias("WINDOW_START"),
        col("COLLECTION_END").alias("WINDOW_END"), 
        col(f"FIRST_200_AFTER_{error_status}"),
        col("HOURS_TO_FIRST_200")
    )

In [ ]:
df_telemap = df_all.filter(col("NUMBER_A_IP_10") == "177.154.149.216")
df_telemap.show()

In [ ]:
# === Aplicação da função
resultado_atual = analyze_recovery_sliding_window_first_only(
    df_all=df_telemap,
    error_status=487,
    collection_window_hours=1,
    min_errors_threshold=2
)

metrics_atual = calculate_recovery_metrics(resultado_atual, df_telemap, 487)
metrics_atual.show()

result_2 = analyze_recovery_sliding_window_first_only(
    df_all=df_telemap,
    error_status=487,
    collection_window_hours=2,
    min_errors_threshold=2
)

metrics_2 = calculate_recovery_metrics(result_2, df_telemap, 487)
metrics_2.show()

result_3 = analyze_recovery_sliding_window_first_only(
    df_all=df_telemap,
    error_status=487,
    collection_window_hours=5,
    min_errors_threshold=2
)

metrics_3 = calculate_recovery_metrics(result_3, df_telemap, 487)
metrics_3.show()

In [ ]:
w60 = (
    Window
    .partitionBy("IDENTIFICATIONNUMBER_B_11")
    .orderBy(col("epoch_min"))
    .rangeBetween(-60, 0)
)

w120 = (
    Window
    .partitionBy("IDENTIFICATIONNUMBER_B_11")
    .orderBy(col("epoch_min"))
    .rangeBetween(-120, 0)
)

w180 = (
    Window
    .partitionBy("IDENTIFICATIONNUMBER_B_11")
    .orderBy(col("epoch_min"))
    .rangeBetween(-180, 0)
)

# Aplicar as três janelas no DataFrame
df_487 = (
    df_all
    .with_column(
        "has_2x487_60min",
        (ssum((col("SIPSTATUSCODE_135") == 487).cast("INTEGER")).over(w60) >= 2)
    )
    .with_column(
        "has_2x487_120min",
        (ssum((col("SIPSTATUSCODE_135") == 487).cast("INTEGER")).over(w120) >= 2)
    )
    .with_column(
        "has_2x487_180min",
        (ssum((col("SIPSTATUSCODE_135") == 487).cast("INTEGER")).over(w180) >= 2)
    )
)

# Visualizar resultado
df_487.select(
    "IDENTIFICATIONNUMBER_B_11", "EPOCH_MIN", "SIPSTATUSCODE_135",
    "has_2x487_60min", "has_2x487_120min", "has_2x487_180min"
).show()

In [ ]:
# Filtros para cada janela de tempo
df_filtrado_60 = df_404.filter(
    ((col("SIPSTATUSCODE_135") == 404) & (col("has_2x404_60min") == True)) |
    ((col("SIPSTATUSCODE_135") == 200) & (col("has_2x404_60min") == False))
)

df_filtrado_120 = df_404.filter(
    ((col("SIPSTATUSCODE_135") == 404) & (col("has_2x404_120min") == True)) |
    ((col("SIPSTATUSCODE_135") == 200) & (col("has_2x404_120min") == False))
)

df_filtrado_180 = df_404.filter(
    ((col("SIPSTATUSCODE_135") == 404) & (col("has_2x404_180min") == True)) |
    ((col("SIPSTATUSCODE_135") == 200) & (col("has_2x404_180min") == False))
)

In [ ]:
# Contagem direta de True em cada coluna
contagens = df_404.select(
    ssum(col("has_2x404_60min").cast("INTEGER")).alias("true_60min"),
    ssum(col("has_2x404_120min").cast("INTEGER")).alias("true_120min"),
    ssum(col("has_2x404_180min").cast("INTEGER")).alias("true_180min")
)

contagens.show()

In [ ]:
df_filtrado.write.mode("overwrite").save_as_table("F_VOICETRAFFIC_SBC_TEMP")

In [ ]:
df_fitrado = session.table("DWDEV.DATASCIENCE.F_VOICETRAFFIC_SBC_TEMP")
df_fitrado

In [ ]:
from snowflake.snowpark import Window
from snowflake.snowpark.functions import col, lit, lag, row_number, datediff, avg, call_function, stddev_samp

# 1) Identificar o INÍCIO do bloqueio por cliente (primeira True após False)
w = Window.partitionBy("IDENTIFICATIONNUMBER_B_11").orderBy(col("DATAHORA"))
df_block_starts = (
    df_filtrado_60
    .with_column("prev_flag", lag(col("HAS_2X404_60MIN"), 1).over(w))
    .with_column(
        "is_block_start",
        (col("HAS_2X404_60MIN") == True) &  # Remover lit() desnecessário
        (col("prev_flag").isNull() | (col("prev_flag") == False))  # isNull() ao invés de is_null()
    )
    .filter(col("is_block_start") == True)  # Remover lit() desnecessário
    .select(
        col("IDENTIFICATIONNUMBER_B_11").alias("CLIENT"),
        col("DATAHORA").alias("BLOCK_TS")
    )
)

# 2) Todas as chamadas 200 do mesmo dataframe
df_200 = (
    df_filtrado_60
    .filter(col("SIPSTATUSCODE_135") == 200)  # Remover lit() desnecessário
    .select(
        col("IDENTIFICATIONNUMBER_B_11").alias("CLIENT"),
        col("DATAHORA").alias("TS_200")
    )
)

# 3) Para cada bloqueio, pegar a PRIMEIRA 200 após o BLOCK_TS
joined = (
    df_block_starts.join(df_200, on=["CLIENT"], how="inner")
                   .filter(col("TS_200") > col("BLOCK_TS"))
)

w2 = Window.partitionBy("CLIENT", "BLOCK_TS").orderBy(col("TS_200"))
first_200 = (
    joined
    .with_column("rn", row_number().over(w2))
    .filter(col("rn") == 1)  # Remover lit() desnecessário
    .with_column("MINS_TO_200", datediff("minute", col("BLOCK_TS"), col("TS_200")))
    .select("CLIENT", "BLOCK_TS", "TS_200", "MINS_TO_200")
)

# Resultados por bloqueio
# first_200.show()

# Estatísticas agregadas (opcional)
stats = first_200.select(
    avg(col("MINS_TO_200")).alias("avg_min"),
    call_function("MEDIAN", col("MINS_TO_200")).alias("median_min"),
    stddev_samp(col("MINS_TO_200")).alias("std_min")
)
# stats.show()

In [ ]:
stats.show()

In [ ]:
from snowflake.snowpark.functions import (
    col, lit, row_number, datediff, avg, call_function, stddev_samp, dateadd
)
# 1) Selecionar os pontos de bloqueio (df_blocks = seu df filtrado)
#    DATAHORA aqui é o timestamp do 2º 404 (início do bloqueio)

blocks = df_filtred.select(
    col("NUMBER_B_IP_11").alias("client_id"),
    col("DATAHORA").alias("block_ts")
)


In [ ]:
# 2) Selecionar todas as chamadas 200 do conjunto completo (df_all)
cand200 = (
    df_all
    .filter(col("SIPSTATUSCODE_135") == lit(200))
    .select(
        col("NUMBER_B_IP_11").alias("client_id"),
        col("DATAHORA").alias("ts_200")
    )
)

In [ ]:
# 2) Selecionar todas as chamadas 200 do conjunto completo (df_all)
cand200 = (
    df_all
    .filter(col("SIPSTATUSCODE_135") == lit(200))
    .select(
        col("NUMBER_B_IP_11").alias("client_id"),
        col("DATAHORA").alias("ts_200")
    )
)
#--------------------------------------------------------------------
# 3) Juntar e pegar a PRIMEIRA 200 após o bloqueio
#--------------------------------------------------------------------
after_join = (
    blocks.join(cand200, on=["client_id"], how="inner")
          .filter(col("ts_200") > col("block_ts"))
)
w_first200 = Window.partitionBy(col("client_id"), col("block_ts")).orderBy(col("ts_200"))
first200 = (
    after_join
    .with_column("rn", row_number().over(w_first200))
    .filter(col("rn") == 1)
    .with_column("mins_to_200", datediff("minute", col("block_ts"), col("ts_200")))
    .select("client_id", "block_ts", "ts_200", "mins_to_200")
)
#--------------------------------------------------------------------
# 4) Estatísticas (média, mediana, desvio-padrão) em minutos
#   - MEDIAN via call_function para garantir compatibilidade
#--------------------------------------------------------------------
stats_direct = first200.select(
    avg(col("mins_to_200")).alias("avg_min"),
    call_function("MEDIAN", col("mins_to_200")).alias("median_min"),
    stddev_samp(col("mins_to_200")).alias("std_min")
)

blocks4 = blocks.with_column("block_release_ts", dateadd("day", lit(4), col("block_ts")))
after_join_4d = (
    blocks4.join(cand200, on=["client_id"], how="inner")
           .filter(col("ts_200") > col("block_release_ts"))
)
w_first200_4d = Window.partitionBy(col("client_id"), col("block_ts")).orderBy(col("ts_200"))
first200_4d = (
    after_join_4d
    .with_column("rn", row_number().over(w_first200_4d))
    .filter(col("rn") == 1)
    .with_column("mins_to_200_from_release",
                 datediff("minute", col("block_release_ts"), col("ts_200")))
    .select("client_id", "block_ts", "block_release_ts", "ts_200", "mins_to_200_from_release")
)
stats_4d = first200_4d.select(
    avg(col("mins_to_200_from_release")).alias("avg_min"),
    call_function("MEDIAN", col("mins_to_200_from_release")).alias("median_min"),
    stddev_samp(col("mins_to_200_from_release")).alias("std_min")
)
# first200_4d.show()
# stats_4d.show()